In [1]:
from transformers import GPT2LMHeadModel


/Users/iman/anaconda3/envs/genai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
checkpoint = "gpt2" # 124M
model_hf = GPT2LMHeadModel.from_pretrained(checkpoint)
sd_hf = model_hf.state_dict()

In [3]:
for k, v in sd_hf.items():
    print(k, v.shape)
#  transformer.wte.weight torch.Size([50257, 768]) ==> 50257 tokens and 768 embedding size   
# transformer.wpe.weight torch.Size([1024, 768]) ==> 1024 is context(batch) size

transformer.wte.weight torch.Size([50257, 768])
transformer.wpe.weight torch.Size([1024, 768])
transformer.h.0.ln_1.weight torch.Size([768])
transformer.h.0.ln_1.bias torch.Size([768])
transformer.h.0.attn.c_attn.weight torch.Size([768, 2304])
transformer.h.0.attn.c_attn.bias torch.Size([2304])
transformer.h.0.attn.c_proj.weight torch.Size([768, 768])
transformer.h.0.attn.c_proj.bias torch.Size([768])
transformer.h.0.ln_2.weight torch.Size([768])
transformer.h.0.ln_2.bias torch.Size([768])
transformer.h.0.mlp.c_fc.weight torch.Size([768, 3072])
transformer.h.0.mlp.c_fc.bias torch.Size([3072])
transformer.h.0.mlp.c_proj.weight torch.Size([3072, 768])
transformer.h.0.mlp.c_proj.bias torch.Size([768])
transformer.h.1.ln_1.weight torch.Size([768])
transformer.h.1.ln_1.bias torch.Size([768])
transformer.h.1.attn.c_attn.weight torch.Size([768, 2304])
transformer.h.1.attn.c_attn.bias torch.Size([2304])
transformer.h.1.attn.c_proj.weight torch.Size([768, 768])
transformer.h.1.attn.c_proj.bias 

### Build GPT Class
- Build the GPTConfig class: to shape the structure of the GPT and define configuration of everything
- Build Multi-head attention class: to creat multi-head attention mechanism in a transformer block 
- Build Multi layer preceptron class: to creat multi-layer preceptron in a transformer block
- Build Block class: to aggregate two previous class to gether plus adding normalization layer and residual path
- Build GPT class: to creat a GPT model instance according to a config, get input sequence and gives next token logits.  

In [4]:
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
import math

In [5]:
# set config as the config of gpt2-124M
@dataclass
class GPTConfig:
    block_size: int = 1024 # max sequence length
    vocab_size: int = 50257 # number of tokens; 50000 BPE + 256 bytes tokens + 1 <|endoftext|> token 
    n_layer: int = 12 # number of layers
    n_head: int = 12 # number of heads
    n_embd: int = 768  # Embedding dimention


    # A dataclass is a Python feature from the dataclasses module (introduced in Python 3.7) 
    # that automatically generates common methods for classes, such as:
    # __init__() (constructor),
    # __repr__() (string representation),
    # __eq__() (equality comparison),
    # and others (depending on parameters like frozen, order, etc.).

    # @dataclass Decorator:
    # When you put @dataclass above a class:
    # It tells Python to automatically create an __init__() method based on the class variables.

    #equvalent to:
    # class GPTConfig:
    #     def __init__(self, block_size=256, vocab_size=65, n_layer=6, n_head=6, n_embed=384):
    #         self.block_size = block_size
    #         self.vocab_size = vocab_size
    #         self.n_layer = n_layer
    #         self.n_head = n_head
    #         self.n_embed = n_embed

In [6]:
# Multi-head attention class
class CasualSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0

        # key, query, value projection for all heads, but in a batch 
        self.c_attn = nn.Linear(config.n_embd, config.n_embd * 3)

        # output projection: is used to just not concating output of heads which might not be in the most useful 
        # format but self.proj allows the model to learn how to best combine the information from different heads
        self.c_proj = nn.Linear(config.n_embd, config.n_embd) # here in this setting n_head * head_size = config.n_embed
        
        # regularization
        self.n_head = config.n_head
        self.n_embd = config.n_embd

        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                             .view(1,1, config.block_size, config.block_size)) # reshape the tril to comply with
                                # shape of (batch, head, block_size, block_size)

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)
    # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        qkv = self.c_attn(x)   # (B, T, 3*C); C = n_embed
        q, k, v = qkv.split(self.n_embd, dim = 2) # each has shape of (B, T, C)

        # creat multi-head. the chanels of each sequence in each batch breaks into nh heads for q, k, v
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)   convert C to nh,hs() and use transpose               
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1))) #(B, nh, T, T)
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side
        # output projection
        y = self.c_proj(y)
        return y


# The Multi-linear preceptron class
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.c_fc = nn.Linear(config.n_embd, config.n_embd * 4)
        self.gelu = nn.GELU(approximate="tanh")
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x


# The transformer block class; contains: 
# normalization + multi-attention + residual pass --> normalization + multi-linear preceptron + residual pass
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.ln_1 = nn.LayerNorm(config.n_embd) # in the paper this layer is after attention layer. 
                                                # but in new design of transformers it is before it
        self.attn = CasualSelfAttention(config)
        
        self.ln_2 = nn.LayerNorm(config.n_embd) # in the paper this layer is after linear layer. 
                                                # but in new design of transformers it is before it
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x)) # In the paper normalization is in the residual pathways. However
                                        # in new design of transformers we have clean residual pathway
        x = x + self.mlp(self.ln_2(x)) 

        return x   



# The GPT class: contains:
# word token embed + position embeding --> 
# multi-transformer blocks + additional normalization layer at the end of blocks -->
# a head to crate logits
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(                        #dict(key1 = 1, key2=2)  # keys are strings
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd)  # additional normalization layer after all blocks before 
                                                 # the head of transformer. which is not in the paper

            )) 
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)   

    def forward(self, idx, targets = None):
        # idx is input to the forward function. idx is shape of (B, T)
        B, T = idx.size()
        assert T <= self.config.block_size, f"Cannot forward sequence of length{T}, block size is less than{self.config.block_size}" 
        # ------- forward the token and position embedding
        # creat positions
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        # get token and position embeddings
        tok_emb = self.transformer.wte(idx) # B, T, n_embd 
        pos_emb = self.transformer.wpe(pos) # T, n_embd
        x = tok_emb + pos_emb
        # -------- forward of the blocks of the transformer 
        for block in self.transformer.h:
            x = block(x)
        # -------- forward the last norm layer and transformer head 
        x = self.transformer.ln_f(x)   
        logits =  self.lm_head(x) #(B, T , Vocab_size)
       
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))    
        return logits, loss

    
    #The @classmethod decorator in Python modifies the method so that it receives the class itself (cls) 
    # as its first argument instead of the instance (self). This is useful when you want to define a method that 
    # acts on the class, not a particular object instance.

    # this method works on the GPT class, and defined to create a model instance from GPT class that loads
    # weights from hugging face
    @classmethod
    def from_pretrained(cls, model_type):
        """Loads pretrained GPT-2 model weights from huggingface"""
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        from transformers import GPT2LMHeadModel
        print("loading weights from pretrained gpt: %s" % model_type)

        # n_layer, n_head and n_embd are determined from model_type
        config_args = {
            'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
            'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
            'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
            'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
        }[model_type]
        config_args['vocab_size'] = 50257 # always 50257 for GPT model checkpoints
        config_args['block_size'] = 1024 # always 1024 for GPT model checkpoints
        # create a from-scratch initialized minGPT model
        config = GPTConfig(**config_args)
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = sd.keys()
        sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # discard this mask / buffer, not a param

        # init a huggingface/transformers model
        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()

        # copy while ensuring all of the parameters are aligned and match in names and shapes
        sd_keys_hf = sd_hf.keys()
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # ignore these, just a buffer
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] # same, just the mask (buffer)
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        # basically the openai checkpoints use a "Conv1D" module, but we only want to use a vanilla Linear
        # this means that we have to transpose these weights when we import them
        assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                # special treatment for the Conv1D weights we need to transpose
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # vanilla copy over the other parameters
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])

        return model    

model = GPT.from_pretrained("gpt2")
print("Didn't crash")

loading weights from pretrained gpt: gpt2
Didn't crash


### Creat GPT2 model by using weight of the model in hugging face (No need to train the model)
- Manage device that model runs on
- Define number of sequences and the length of them that the model should creat for each input sequence
- Get input sequence and tokenize it
- Forward the tokenized sequence to the model to get next token while it reaches the maximum length(Generate text)
- Decode the final outputs and print them 

In [7]:
device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'

print(f"Using device is: {device}")

# number of sequences that should be returned
num_return_sequences = 5
# maximum length of each sequence
max_length = 30

# instanciate model from GPT class with loaded weights from hugging face 
model = GPT.from_pretrained('gpt2')

# put model in evaluation phase. It is good practice you put the model on eval mode when you are not 
# training it,  you are just using it.
# right now we do not have any layer or module that have different behavior at training or evaluating time.
# like: dropout, batchnorm. but it is a good practice, especially it might have effect on torch internals 
# although all layer or module are the same at training and evaluation time
model.eval() # as GPT inheritate from torch.nn.Module; all methods in it works on any instances from it

# put model on available device
model.to(device)

Using device is: mps
loading weights from pretrained gpt: gpt2


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): CasualSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='tanh')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [8]:
# create idx; the input to the model
text = "Hello, I'm a language model,"
import tiktoken
enc = tiktoken.get_encoding('gpt2')
tokens = enc.encode(text)
tokens = torch.tensor(tokens, dtype=torch.long) # shape (8,)
tokens = tokens.unsqueeze(0).repeat(num_return_sequences, 1) # shape (1, 8) --> repeated 5 times gives --> 5,8
x = tokens.to(device)

In [9]:
# ----- Generate text
# x is (B,T) where B = 5 and T is 8
torch.manual_seed(42)
while x.size(1) <= max_length:
    with torch.no_grad():
        #forward x through the model to get logits of last time (T) across all batches
        logits, _ = model(x) #(B, T, vocab_size)
        logits = logits[:, -1, :] # (B, vocab_size)
        
        # convert logits to probabilities
        probability = F.softmax(logits, dim= -1) # (B, vocab_size)

        # get top 50 probability indices; according to hugging face approach
        probs, indices = torch.topk(probability, 50, dim= -1) # (B, 50)

        # get the most probable 
        ix = torch.multinomial(probs, 1) #(B, 1)

        # get the index of most probable one as next token
        xcol = torch.gather(indices, -1, ix) #(B, 1)

        # append to the sequence
        x = torch.cat((x, xcol), dim=1)  # at the end x is (B, max_length)

# print the generated text
for i in range(num_return_sequences):
    tokens = x[i,:max_length].tolist()
    decoded = enc.decode(tokens)
    print(">", decoded)

> Hello, I'm a language model, which means I'm familiar with it, but I'm not fluent in that. Well, with that said,
> Hello, I'm a language model, and the syntax, to make use of it, is pretty good. So why do you have that and not
> Hello, I'm a language model, I'm doing this work in Python, and then I'm writing code for Haskell.

So we can
> Hello, I'm a language model, and you're making assumptions about my use of them. I'm not a natural language learner. I'm
> Hello, I'm a language model, well, I'm from Java and have to write a programming language for it. I have my own vocabulary because


### Train the GPT from scratch

##### Load Data

In [10]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

data = text[:1000]
print("length of dataset in characters: ", len(text))
# let's look at the first 100 characters
print(data[:100])    

--2025-05-28 10:28:26--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8000::154, 2606:50c0:8003::154, 2606:50c0:8001::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8000::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.2’

input.txt.2         100%[===================>]   1.06M  --.-KB/s    in 0.07s   

2025-05-28 10:28:27 (15.9 MB/s) - ‘input.txt.2’ saved [1115394/1115394]

length of dataset in characters:  1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


##### Implementing a toy training by just one batch

In [ ]:
device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'

print(f"Using device is: {device}")

#get data batch
import tiktoken
enc = tiktoken.get_encoding('gpt2')
tokens = enc.encode(data)
B, T = 4, 32
buf = torch.tensor(tokens[: B*T + 1])
buf = buf.to(device)
x = buf[:-1].view(B,T)
y = buf[1:].view(B,T)


model = GPT(GPTConfig())
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

for i in range(50):
    optimizer.zero_grad() # it is a must to call this before backward path, and good practice to even call it before forward path
    
    logits, loss = model(x,y)   # in the first ittereation the loss is expected to be around -ln(1/50257) which is 10.8249 
                                # Note, loss is: average of sum of all -ln(p(xi|yi)) = (32*4)*-ln(1/50257)/(32*4)   
    loss.backward()
    optimizer.step()
    print(f"step {i}, loss: {loss.item()} ")



Using device is: mps
step 0, loss: 10.99524974822998 
step 1, loss: 10.735877990722656 
step 2, loss: 10.429094314575195 
step 3, loss: 10.251737594604492 
step 4, loss: 10.186267852783203 
step 5, loss: 10.199043273925781 
step 6, loss: 10.16696548461914 
step 7, loss: 9.983144760131836 
step 8, loss: 9.75760269165039 
step 9, loss: 9.548858642578125 
step 10, loss: 9.347602844238281 
step 11, loss: 9.15156364440918 
step 12, loss: 8.968486785888672 
step 13, loss: 8.801366806030273 
step 14, loss: 8.643999099731445 
step 15, loss: 8.48674201965332 
step 16, loss: 8.322237014770508 
step 17, loss: 8.147783279418945 
step 18, loss: 7.966440677642822 
step 19, loss: 7.785323619842529 
step 20, loss: 7.6119489669799805 
step 21, loss: 7.450068473815918 
step 22, loss: 7.298414707183838 
step 23, loss: 7.153176307678223 
step 24, loss: 7.010628700256348 
step 25, loss: 6.868446350097656 
step 26, loss: 6.726003646850586 
step 27, loss: 6.5836944580078125 
step 28, loss: 6.441592216491699 